In [0]:
from typing import Dict, List, Tuple
import logging
from datetime import datetime

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


def run_notebook(notebook_path: str, notebook_name: str) -> Tuple[bool, str]:
    """
    Execute a notebook using %run command with error handling.
    
    Args:
        notebook_path: Full path to the notebook
        notebook_name: Display name for logging
    
    Returns:
        Tuple of (success: bool, message: str)
    """
    try:
        logger.info(f"Starting execution: {notebook_name}")
        logger.info(f"Path: {notebook_path}")
        
        # Execute the notebook
        dbutils.notebook.run(notebook_path, timeout_seconds=600)
        
        logger.info(f"✓ Successfully completed: {notebook_name}")
        return True, f"Success: {notebook_name}"
        
    except Exception as e:
        error_msg = f"✗ Failed to execute {notebook_name}: {str(e)}"
        logger.error(error_msg, exc_info=True)
        return False, error_msg


def execute_etl_pipeline(stop_on_error: bool = True) -> Dict[str, any]:
    """
    Execute the complete ETL pipeline in sequence.
    
    Pipeline stages:
    1. Bronze Layer: Load raw files (CCD XML, Claims CSV)
    2. Silver Layer: Transform medications
    3. Silver Layer: Transform problems
    4. Gold Layer: Consumption layer aggregations
    
    Args:
        stop_on_error: If True, stops pipeline on first error. If False, continues and reports all errors.
    
    Returns:
        Dictionary with execution results and statistics
    """
    start_time = datetime.now()
    
    # Define pipeline stages
    pipeline_stages = [
        {
            "name": "Bronze - Load Raw Files",
            "path": "/Repos/harbir.mahal@gmail.com/etl/ingestion/Bronze/load_raw_file",
            "layer": "bronze"
        },
        {
            "name": "Silver - Transform Medications",
            "path": "/Repos/harbir.mahal@gmail.com/etl/transformation/Silver/transform_medications",
            "layer": "silver"
        },
        {
            "name": "Silver - Transform Problems",
            "path": "/Repos/harbir.mahal@gmail.com/etl/transformation/Silver/transform_problems",
            "layer": "silver"
        },
        {
            "name": "Gold - Consumption Layer",
            "path": "/Repos/harbir.mahal@gmail.com/etl/transformation/Gold/consumption",
            "layer": "gold"
        }
    ]
    
    results = {
        "start_time": start_time,
        "stages": [],
        "total_stages": len(pipeline_stages),
        "successful_stages": 0,
        "failed_stages": 0,
        "status": "running"
    }
    
    logger.info("="*70)
    logger.info("STARTING ETL PIPELINE EXECUTION")
    logger.info(f"Total stages: {len(pipeline_stages)}")
    logger.info(f"Stop on error: {stop_on_error}")
    logger.info("="*70)
    
    # Execute each stage
    for idx, stage in enumerate(pipeline_stages, 1):
        stage_start = datetime.now()
        
        logger.info("")
        logger.info(f"[{idx}/{len(pipeline_stages)}] Executing: {stage['name']}")
        logger.info("-" * 70)
        
        success, message = run_notebook(stage["path"], stage["name"])
        
        stage_end = datetime.now()
        stage_duration = (stage_end - stage_start).total_seconds()
        
        stage_result = {
            "stage_number": idx,
            "name": stage["name"],
            "layer": stage["layer"],
            "path": stage["path"],
            "success": success,
            "message": message,
            "duration_seconds": stage_duration
        }
        
        results["stages"].append(stage_result)
        
        if success:
            results["successful_stages"] += 1
            logger.info(f"✓ Stage completed in {stage_duration:.2f} seconds")
        else:
            results["failed_stages"] += 1
            logger.error(f"✗ Stage failed after {stage_duration:.2f} seconds")
            
            if stop_on_error:
                logger.error("Stopping pipeline due to error (stop_on_error=True)")
                results["status"] = "failed"
                break
    
    # Calculate final statistics
    end_time = datetime.now()
    total_duration = (end_time - start_time).total_seconds()
    
    results["end_time"] = end_time
    results["total_duration_seconds"] = total_duration
    
    if results["failed_stages"] == 0:
        results["status"] = "success"
    elif results["status"] != "failed":  # Not already marked as failed by stop_on_error
        results["status"] = "partial_success"
    
    # Print summary
    print_pipeline_summary(results)
    
    return results


def print_pipeline_summary(results: Dict[str, any]):
    """
    Print a formatted summary of the pipeline execution.
    
    Args:
        results: Dictionary containing execution results
    """
    print("\n")
    print("="*70)
    print("ETL PIPELINE EXECUTION SUMMARY")
    print("="*70)
    print(f"Status: {results['status'].upper()}")
    print(f"Total Duration: {results['total_duration_seconds']:.2f} seconds")
    print(f"Successful Stages: {results['successful_stages']}/{results['total_stages']}")
    print(f"Failed Stages: {results['failed_stages']}/{results['total_stages']}")
    print("\n")
    
    print("-"*70)
    print("STAGE DETAILS:")
    print("-"*70)
    
    for stage in results["stages"]:
        status_icon = "✓" if stage["success"] else "✗"
        status_text = "SUCCESS" if stage["success"] else "FAILED"
        
        print(f"\n[{stage['stage_number']}] {stage['name']}")
        print(f"    Status: {status_icon} {status_text}")
        print(f"    Duration: {stage['duration_seconds']:.2f}s")
        print(f"    Layer: {stage['layer']}")
        
        if not stage["success"]:
            print(f"    Error: {stage['message']}")
    
    print("\n" + "="*70)
    
    if results["status"] == "success":
        print("✓ ✓ ✓  ALL STAGES COMPLETED SUCCESSFULLY  ✓ ✓ ✓")
    elif results["status"] == "partial_success":
        print("⚠ PIPELINE COMPLETED WITH SOME FAILURES")
    else:
        print("✗ PIPELINE FAILED")
    
    print("="*70)
    print("\n")


# Main execution
if __name__ == "__main__":
    # Execute the pipeline
    results = execute_etl_pipeline(stop_on_error=True)
    
    # Return results for downstream use
    pipeline_results = results